# F-Audit: Phase 2 form test on h/m movies

**Date:** 2026-04-18.
**Purpose:** Test the phase 1 / phase 2 decomposition using only movies where h/m close-day timestamps make both phases observable. Compares `constant=C` vs `F × mean_close_day_count` for Phase 2.

**Clean decomposition (per brainstorm):**
- **Phase 1:** KDE predicts `snap → midnight UTC of close day`. Integration window: `(midnight_utc_dbc, snap_dbc]`.
- **Phase 2:** covers the 14-hour `midnight UTC → 10am EDT` window. Observable ONLY on h/m movies.

**Why only h/m targets:** for day-level movies, all close-day reviews floor to midnight UTC, so we cannot observe the true pre-market (12am-10am EDT) count. For the 5 h/m targets, we have exact timestamps on close day and can cleanly split actual into phase1 and phase2 components.

**Targets:** `the_drama`, `the_super_mario_galaxy_movie`, `forbidden_fruits_2026`, `they_will_kill_you`, `you_me_and_tuscany`.

**Note on scraper coverage:**
- `the_drama` and `the_super_mario_galaxy_movie`: scraper disabled before market close → actual_phase2 may be under-counted (missed post-midnight reviews before scraper stopped). Included for completeness but flagged.
- `forbidden_fruits`, `they_will_kill_you`, `you_me_and_tuscany`: scraper stayed on past close → clean actual_phase2 counts.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name != 'notebooks':
    ROOT = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from _helpers import (
    reviews, movies, close_date_map, gaps, gap_lookup, first_review_ts,
    gap_for_slug,
    combined_score_selector,
    snapshot_state, actual_remaining, close_day_count,
    build_critic_profiles, build_kde_lambda_model_capped, predict_window,
    passes_skip_rules_for_snap,
)

# Ship stack parameters
SHIP_ALPHA = 0.5
SHIP_SIGMA_GAP = 8.0
SHIP_N_TRAINING = 20
SHIP_BANDWIDTH_FLOOR = 0.5
SHIP_BANDWIDTH_CEIL = 0.7

TARGETS = [
    'the_drama',
    'the_super_mario_galaxy_movie',
    'forbidden_fruits_2026',
    'they_will_kill_you',
    'you_me_and_tuscany',
]
SNAPS = [1.0, 3.0]

# For each target, compute midnight_utc_dbc (days between midnight UTC of close day and market close)
for t in TARGETS:
    c = close_date_map[t]
    mid_utc = c.floor('D')
    dbc = (c - mid_utc).total_seconds() / 86400
    print(f'{t:32s}  close={c}  midnight_utc_dbc={dbc:.4f}')

## Ground truth: phase 1 and phase 2 actual counts per (target, snap)

In [ ]:
def split_actual(slug, snap_dbc):
    """Return (actual_phase1, actual_phase2) — counts of reviews in each sub-window."""
    close_ts = close_date_map[slug]
    midnight_utc = close_ts.floor('D')
    midnight_utc_dbc = (close_ts - midnight_utc).total_seconds() / 86400

    movie_reviews = reviews[reviews['movie_slug'] == slug].copy()
    movie_reviews['dbc'] = (close_ts - movie_reviews['estimated_timestamp']).dt.total_seconds() / 86400

    # Phase 1 window: (midnight_utc_dbc, snap_dbc]  — i.e., reviews strictly before midnight UTC of close day
    phase1 = ((movie_reviews['dbc'] > midnight_utc_dbc) & (movie_reviews['dbc'] <= snap_dbc)).sum()
    # Phase 2 window: (0, midnight_utc_dbc]  — i.e., pre-market close-day reviews
    phase2 = ((movie_reviews['dbc'] > 0) & (movie_reviews['dbc'] <= midnight_utc_dbc)).sum()
    return int(phase1), int(phase2)

rows = []
for target in TARGETS:
    for snap in SNAPS:
        a1, a2 = split_actual(target, snap)
        rows.append({
            'target': target, 'snap_dbc': snap,
            'actual_phase1': a1,
            'actual_phase2': a2,
            'actual_total': a1 + a2,
            'actual_remaining_lib': actual_remaining(target, snap),  # sanity: should == a1+a2 if no day-level close-day
        })
truth = pd.DataFrame(rows)
print('Ground truth per (target, snap):')
print(truth.to_string(index=False))
# Sanity: actual_remaining_lib should match actual_total (since no day-level close-day reviews)
mismatch = truth[truth['actual_total'] != truth['actual_remaining_lib']]
if len(mismatch):
    print(f'\nWARNING: {len(mismatch)} mismatches between library actual_remaining and our a1+a2:')
    print(mismatch.to_string(index=False))
else:
    print('\nSanity check: library actual_remaining = actual_phase1 + actual_phase2 for all rows (no day-level close-day reviews in this subset).')

## Phase 1 prediction: KDE integral `(midnight_utc_dbc, snap_dbc]`

Uses ship stack: `combined_score(α=0.5, σ_gap=8) + ceil=0.7 + n=20`, with snap-adaptive skip rule.

In [ ]:
def run_phase1(target, snap_dbc):
    """Return (phase1_pred, training_slugs, mean_cd_training, skip_reason)."""
    target_gap = gap_for_slug(target)
    if target_gap is None:
        return None, None, None, 'no gap (no reviews)'
    close_ts = close_date_map[target]
    midnight_utc_dbc = (close_ts - close_ts.floor('D')).total_seconds() / 86400
    snap_time = close_ts - pd.Timedelta(days=snap_dbc)
    state = snapshot_state(target, snap_time)
    passed, reason = passes_skip_rules_for_snap(state, snap_dbc)
    if not passed:
        return None, None, None, reason

    target_window_days = state['first_review_dbc'] - snap_dbc
    target_critics = state['observed_critics']

    training, _ = combined_score_selector(
        target, target_gap, target_critics, target_window_days,
        k=SHIP_N_TRAINING, alpha=SHIP_ALPHA, sigma_gap=SHIP_SIGMA_GAP,
    )
    if len(training) < 5:
        return None, None, None, f'too few training slugs ({len(training)})'

    profiles = build_critic_profiles(reviews, close_date_map, training, verbose=False)
    model = build_kde_lambda_model_capped(
        profiles,
        bandwidth_floor=SHIP_BANDWIDTH_FLOOR,
        bandwidth_ceiling=SHIP_BANDWIDTH_CEIL,
    )

    # Phase 1: integrate KDE over (midnight_utc_dbc, snap_dbc]
    phase1 = predict_window(
        model, dbc_from=snap_dbc, dbc_to=midnight_utc_dbc,
        observed_critics=target_critics,
        observed_count=state['observed_count'],
        first_review_dbc=state['first_review_dbc'],
    )
    mean_cd = float(np.mean([close_day_count(s) for s in training]))
    return float(phase1), training, mean_cd, None

rows = []
for target in TARGETS:
    for snap in SNAPS:
        p1, training, mean_cd, reason = run_phase1(target, snap)
        rows.append({
            'target': target, 'snap_dbc': snap,
            'phase1_pred': p1, 'mean_cd_training': mean_cd,
            'n_training': len(training) if training else 0,
            'skip': reason,
        })
preds = pd.DataFrame(rows)
print('Phase 1 predictions per (target, snap):')
print(preds.to_string(index=False, float_format='%.2f'))

## Per-target decomposition: phase 1 predicted vs actual

How well does the KDE alone (phase 1, no close-day extrapolation) track reviews in `(midnight_utc_dbc, snap_dbc]`?

In [ ]:
combined = truth.merge(preds, on=['target', 'snap_dbc'])
combined['phase1_err'] = combined['phase1_pred'] - combined['actual_phase1']
# MAE on phase 1 alone
print('Phase 1 alone (KDE integral on observable window):')
print(combined[['target', 'snap_dbc', 'actual_phase1', 'phase1_pred', 'phase1_err']]
      .to_string(index=False, float_format='%.2f'))
print()
for snap in SNAPS:
    sub = combined[combined['snap_dbc'] == snap].dropna(subset=['phase1_pred'])
    if len(sub):
        mae = sub['phase1_err'].abs().mean()
        mean_err = sub['phase1_err'].mean()
        print(f'  T-{snap:g}d  n={len(sub)}  MAE={mae:.2f}  mean_err={mean_err:+.2f}')

## Phase 2 sweep: which form best predicts close-day pre-market count?

For each target, try:
- **Constant lambda:** `C ∈ {0, 1, 2, 3}`
- **Proportional:** `F × mean_close_day_count(training)` for `F ∈ {0.2, 0.7}` (previous ship candidates)

Compare to the ground-truth `actual_phase2` per target.

In [ ]:
phase2_forms = []
for C in [0, 1, 2, 3]:
    phase2_forms.append(('constant', f'C={C}', lambda row, c=C: float(c)))
for F in [0.2, 0.7]:
    phase2_forms.append(('proportional', f'F={F:g}×mean_cd', lambda row, f=F: f * row['mean_cd_training']))

print('Phase 2 prediction vs actual_phase2 per target:')
print(f'{"target":32s} {"snap":5s} {"actual":6s}   ' + '  '.join(f'{label:11s}' for _, label, _ in phase2_forms))
for _, row in combined.iterrows():
    if row['skip']:
        continue
    preds_p2 = [fn(row) for _, _, fn in phase2_forms]
    pred_str = '  '.join(f'{p:11.2f}' for p in preds_p2)
    print(f'{row["target"]:32s} T-{row["snap_dbc"]:g}d  {row["actual_phase2"]:5d}    {pred_str}')

In [ ]:
# Full-window MAE: phase1_pred + phase2_pred vs actual_total
print('FULL WINDOW MAE (phase1 + phase2 vs actual_total), split by snap:')
print()
for snap in SNAPS:
    sub = combined[(combined['snap_dbc'] == snap) & combined['phase1_pred'].notna()].copy()
    if not len(sub):
        continue
    print(f'--- T-{snap:g}d snap (n={len(sub)}) ---')
    header = f'  {"phase2_form":16s}  {"MAE":>6s}  {"mean_err":>9s}  {"median_err":>11s}'
    print(header)
    for kind, label, fn in phase2_forms:
        preds_total = sub['phase1_pred'] + sub.apply(fn, axis=1)
        err = preds_total - sub['actual_total']
        mae = err.abs().mean()
        print(f'  {label:16s}  {mae:6.2f}  {err.mean():+9.2f}  {err.median():+11.2f}')
    print()

## Flagged subset: drop scraper-capped movies

`the_drama` and `the_super_mario_galaxy_movie` had the scraper disabled before market close. Their `actual_phase2` under-counts the true pre-market arrivals (they would have seen post-scraper-shutoff activity that wasn't captured). Re-run the phase-2 sweep on just the 3 usable movies.

In [ ]:
CLEAN_TARGETS = ['forbidden_fruits_2026', 'they_will_kill_you', 'you_me_and_tuscany']
clean = combined[combined['target'].isin(CLEAN_TARGETS)].copy()

print(f'CLEAN SUBSET ({len(CLEAN_TARGETS)} movies, scraper stayed past close):')
for snap in SNAPS:
    sub = clean[(clean['snap_dbc'] == snap) & clean['phase1_pred'].notna()].copy()
    if not len(sub):
        continue
    print(f'\n--- T-{snap:g}d (n={len(sub)}) ---')
    print(f'  {"phase2_form":16s}  {"MAE":>6s}  {"mean_err":>9s}')
    for kind, label, fn in phase2_forms:
        preds_total = sub['phase1_pred'] + sub.apply(fn, axis=1)
        err = preds_total - sub['actual_total']
        mae = err.abs().mean()
        print(f'  {label:16s}  {mae:6.2f}  {err.mean():+9.2f}')

print('\nPer-target decomposition (clean subset):')
print(clean[['target', 'snap_dbc', 'actual_phase1', 'phase1_pred', 'actual_phase2', 'mean_cd_training']]
      .to_string(index=False, float_format='%.2f'))

## Takeaways

Read the full-window MAE tables above for each snap. Key questions:

1. **Is phase 1 systematically under/over?** Look at `mean_err` for the `C=0` row — this isolates phase 1's quality.
2. **Is constant better than proportional?** Compare `C=2` vs `F=0.2×mean_cd` MAE at each snap.
3. **Does the answer differ by snap?** T-1d is the piecewise's intended use case (close-day dynamics dominate). T-3d is where the KDE has more work to do.

With n=5 (or n=3 clean), these numbers are directional. A formal ship decision on the phase-2 form needs more h/m targets — but this gives us a first read on whether constant is in the ballpark.